# Crypto Trading Signal Bot — Colab Quickstart

Runs the bot straight from GitHub inside Google Colab.

**Colab is good for:** testing the setup, seeing what signals look like, tuning
parameters, exploring indicator values.

**Colab is NOT good for 24/7 alerts.** A Colab runtime disconnects after roughly
90 minutes idle and is capped around 12 hours even when active, the browser tab
has to stay open, and every disconnect wipes `signal_state.json` (so you get
duplicate alerts on the next run). For always-on alerts use Railway, a VPS or
Docker — see the README.

Run the cells top to bottom.


## 1. Get the code and install dependencies

**If this repo is private**, Colab cannot clone it anonymously. Either make the
repo public (Settings -> General -> Danger Zone -> Change visibility — there are
no secrets in it; your Telegram token lives only in environment variables), or
create a fine-grained token with **Contents: Read-only** at
[github.com/settings/personal-access-tokens](https://github.com/settings/personal-access-tokens)
and paste it when the cell asks.

The token is read with `getpass`, never printed, and removed from the git remote
right after cloning — so it cannot leak into saved notebook output.


In [ ]:
import os, shutil, subprocess
from getpass import getpass

REPO   = 'samshoaib123/Trading_Signal_Bot'
BRANCH = 'claude/crypto-trading-signal-bot-xmcatj'   # use 'main' once merged

token = getpass('GitHub token (just press Enter if the repo is public): ').strip()
auth_url  = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
clean_url = f'https://github.com/{REPO}.git'

if os.path.exists('Trading_Signal_Bot'):
    shutil.rmtree('Trading_Signal_Bot')      # start clean on a re-run

# subprocess, not '!git clone', so a failure message can never echo the token.
result = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, auth_url, 'Trading_Signal_Bot'],
    capture_output=True, text=True,
)

if result.returncode != 0:
    stderr = result.stderr.replace(token, '***') if token else result.stderr
    print('Clone failed:\n', stderr)
    print('\nMost likely causes:')
    print('  - the repo is private and you left the token blank')
    print('  - the token lacks "Contents: Read-only" for this repository')
    print(f'  - the branch {BRANCH!r} does not exist')
else:
    # Drop the credential from the remote so it is not stored in the runtime.
    subprocess.run(['git', '-C', 'Trading_Signal_Bot', 'remote', 'set-url',
                    'origin', clean_url], check=False)
    del token, auth_url
    %cd Trading_Signal_Bot
    !pip install --quiet -r requirements.txt
    print('\nReady.')


## 2. Enter your Telegram credentials

`getpass` hides the token as you type so it is **not** saved into the notebook
output. Never paste your bot token directly into a cell — notebooks get shared.

- Token: from [@BotFather](https://t.me/BotFather) (`/newbot`)
- Chat ID: from [@userinfobot](https://t.me/userinfobot)
- **Send `/start` to your own bot first**, or Telegram will refuse to deliver.


In [ ]:
import os
from getpass import getpass

os.environ['TELEGRAM_BOT_TOKEN'] = getpass('Telegram bot token: ').strip()
os.environ['TELEGRAM_CHAT_ID'] = input('Telegram chat id: ').strip()

# Optional overrides — uncomment and edit any of these.
# os.environ['SYMBOLS'] = 'BTC/USDT,ETH/USDT,SOL/USDT'
# os.environ['CAPITAL'] = '1000'
# os.environ['RISK_PERCENT'] = '1'
# os.environ['MIN_CONFIDENCE'] = '2'      # only stronger signals
# os.environ['LOG_LEVEL'] = 'DEBUG'

print('Credentials set for this runtime only.')


## 3. Preflight — check everything at once

This verifies config, exchange reachability, candle download, indicators, the
state file and Telegram delivery, and prints a fix for whatever fails.

### ⚠️ Expect a Binance 451 here

Colab runs on Google servers in the US, and **`binance.com` blocks US IP
addresses**. If preflight shows `451` or `restricted location`, that is not a bug
in the bot — just switch exchanges in the next cell. The bot works the same on
any of them.


In [ ]:
!python main.py --preflight


### If you saw a 451, run this cell and then re-run preflight above


In [ ]:
import os
os.environ['EXCHANGE_ID'] = 'kucoin'   # also try: okx, bybit, binanceus
print('Switched to', os.environ['EXCHANGE_ID'], '- re-run the preflight cell.')


## 4. One scan, without sending anything

Fetches real candles and evaluates every setup, but only logs what it *would*
send. Safe way to see the bot working.


In [ ]:
!python main.py --once --dry-run


## 5. One real scan (sends to Telegram)

Alerts only fire on a *fresh* cross on the last closed candle, so an empty run is
normal and correct — it means no setup triggered in the last 15 minutes.


In [ ]:
!python main.py --once


## 6. Explore the numbers yourself

Pull one pair and look at the raw indicator values — useful for sanity-checking
or tuning thresholds before you deploy for real.


In [ ]:
import config, exchange, indicators, strategies

settings = config.load_settings()
config.configure_logging('WARNING')
ex = exchange.create_exchange(settings)

symbol = 'BTC/USDT'
df = exchange.fetch_ohlcv(ex, symbol, settings)
enriched = indicators.calculate_indicators(df, settings)

cols = ['close', 'rsi', 'macd', 'macd_signal', 'macd_hist',
        'bb_lower', 'bb_upper', 'atr']
display(enriched[cols].tail(10).round(4))

found = strategies.detect_signals(symbol, enriched, settings)
print(f'\nSignals on the last closed candle: {len(found)}')
for s in found:
    print(' ', s.side, s.setup_label,
          f'| entry {s.entry:.2f} SL {s.stop_loss:.2f} TP {s.take_profit:.2f}',
          f'| confidence {s.confidence}/3')


## 7. Keep it running (with eyes open)

This loops on the 15-minute candle close until you stop the cell or Colab
disconnects. **Keep the tab open.** Expect the runtime to die within a few hours —
that is Colab, not the bot.

Stop it with the ⏹ button next to the cell.


In [ ]:
!python main.py


---

## Ready for always-on alerts?

Colab cannot do it. Pick one of these instead (all documented in the README):

| Option | Cost | Notes |
|---|---|---|
| Railway | ~$5/month | Easiest. Connect the repo, set two variables. |
| VPS + systemd | ~$4/month | Most control. Setup commands are in the README. |
| Docker | your host | `docker compose up -d` |

Whichever you choose, run `python main.py --preflight` there once — it will tell
you immediately if anything is misconfigured.
